Severity: transformer retraining on the CNN dataset
Kaggle: enable GPU and Internet. This notebook performs one patient-grouped
70/15/15 experiment per task/model, not K-fold. The nine previously successful
architectures are included. Old results and checkpoints are incompatible with
these new splits and label mappings. CPU data checks run before training.
Numbered sections below replace the old notebook's duplicated pipeline.


1. Dependencies
Keep Kaggle's installed PyTorch/CUDA pair. Install the colleague's timm version.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.22', 'gdown==5.2.0', 'openpyxl'], check=True)



2. Configuration
RUN_TRAINING=False permits a CPU-only dataset audit. Change to True to train.
Default all-layer tuning and L2 embeddings retain the colleague's baseline.
Differential learning rates: pretrained backbone 1e-5, new classifier 1e-4.
All layers remain trainable. Cosine decay scales both rates together.
Two rates are a starting configuration, not a validated optimum or a guarantee
against overfitting. Early stopping still monitors patient-held-out validation loss.
Layer-wise decay is a separate option for future experiments: earlier/input
blocks receive smaller rates than later/output blocks, not the reverse.
Research reference: https://github.com/facebookresearch/mae/blob/main/util/lr_decay.py
TUNE_MODE: all, head, last_blocks (ViT/DeiT), last_stages (Swin).
TUNE_OVERRIDES examples: {'ViT-B/16': ('last_blocks', 4),
                        'Swin-Tiny': ('last_stages', 1)}.
These settings are hypotheses, not Optuna-selected winners. Keep the test split
untouched when selecting settings. Set RUN_NAME to a new value after any change.
Local file overrides can point to attached /kaggle/input files instead of Drive.
To resume in a fresh session, restore the complete prior RUN_NAME output folder
to OUTPUT_ROOT before running; resume state is saved after every epoch.


In [ ]:
from pathlib import Path
import os, json, hashlib, zipfile, random, copy, gc, platform
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
DATA_ROOT = ROOT / 'transformer_data_v2'
RUN_NAME = 'transformers_severity_v2_two_lr'
OUTPUT_ROOT = ROOT / 'outputs' / RUN_NAME
RUN_TRAINING = True
TASKS = ['severity']
MODEL_NAMES = ['ViT-B/16', 'ViT-B/32', 'DeiT-Tiny', 'DeiT-Small',
               'DeiT-Base', 'DeiT-Small-Distilled', 'Swin-Tiny', 'Swin-Small', 'Swin-Base']
CONFIG = dict(seed=42, epochs=30, batch_size=32, workers=2, patience=5,
              backbone_lr=1e-5, head_lr=1e-4, weight_decay=0.01,
              tune_mode='all', tune_depth=4, l2_features=True)
TUNE_OVERRIDES = {}
SOURCES = {
    'severity': dict(zip_id='117lqrb69UR41ZeQXtTX895dE3sF8iTmo',
                     metadata_id='1YSX2UP_paaqiCQ9W1mzVKub7GvJDL7i7', ext='xlsx',
                     classes=['low nuclear density', 'dense', 'mature', 'brunescent'],
                     rows=2541, videos=645, counts={0:795, 1:1047, 2:614, 3:85},
                     expected_hash='1bcc1c61c024845835cabc7ad020f6d1128a2a3ff929e03bf0d8fd53ee4b5e50'),
    'poor_dilation': dict(zip_id='1hpCGfGdu-wjB2oi0xFNzb-pF4-mWjbWi',
                          metadata_id='1rHw_AlqBx_wdmsKtlJWlzt8-Hs4ZdSmF', ext='csv',
                          classes=['normal', 'poor dilation'], rows=3081, videos=700,
                          counts={0:2541, 1:540},
                          expected_hash='dd8462f5408f9637e63ab7c8f70325f23a90db6a58da5080f81d1f6033150a4f')}
LOCAL_FILES = {t: {'zip': None, 'metadata': None} for t in SOURCES}



3. Downloads, exact metadata checks, image validation and saved patient splits
The canonical metadata checksum must match the label files supplied for review.
Metadata labels are per frame. Mixed-label poor-dilation videos stay together.
The split stratifies patients by their set of labels (normal / poor / mixed).
This is a NEW single split, not the old transformer split or a CNN K-fold split.


In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def table_hash(df, columns):
    data = df[columns].sort_values(['video_id', 'file_name']).reset_index(drop=True)
    return hashlib.sha256(data.to_csv(index=False, lineterminator='\n').encode()).hexdigest()

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False))
    temp.replace(path)

def fetch(task, kind):
    import gdown, requests
    spec = SOURCES[task]
    override = LOCAL_FILES[task][kind]
    if override:
        p = Path(override)
        require(p.is_file(), f'Missing local file: {p}')
        return p
    p = DATA_ROOT / task / ('images.zip' if kind == 'zip' else 'labels.' + spec['ext'])
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.is_file():
        return p
    part = p.with_suffix(p.suffix + '.part')
    if kind == 'metadata' and spec['ext'] == 'xlsx':
        response = requests.get('https://docs.google.com/spreadsheets/d/' +
                                spec['metadata_id'] + '/export?format=xlsx', timeout=180)
        response.raise_for_status()
        part.write_bytes(response.content)
    else:
        result = gdown.download(id=spec['zip_id'] if kind == 'zip' else spec['metadata_id'],
                                output=str(part), quiet=False)
        require(result is not None, 'Drive download failed. Check sharing or use LOCAL_FILES.')
    require(part.is_file() and part.stat().st_size > 0, f'Empty download: {p}')
    if kind == 'zip' or spec['ext'] == 'xlsx':
        require(zipfile.is_zipfile(part), f'Invalid ZIP/XLSX: {p}')
    part.replace(p)
    return p

def canonical_metadata(path, task):
    spec = SOURCES[task]
    raw = pd.read_excel(path, sheet_name='Dataset_Severity_V2') if spec['ext'] == 'xlsx' else pd.read_csv(path)
    raw.columns = raw.columns.astype(str).str.strip()
    columns = ['filename', 'videoname'] + spec['classes']
    require(set(columns) <= set(raw.columns), 'Missing metadata columns')
    require(not raw[columns].isna().any().any(), 'Missing metadata values')
    labels = raw[spec['classes']]
    require(labels.isin([0, 1]).all().all() and labels.sum(axis=1).eq(1).all(), 'Invalid one-hot labels')
    df = pd.DataFrame(dict(file_name=raw.filename.astype(str).str.strip(),
                           video_id=raw.videoname.astype(str).str.strip(),
                           label=labels.to_numpy().argmax(axis=1)))
    require(df.file_name.ne('').all() and df.video_id.ne('').all(), 'Empty identifiers')
    require(df.groupby(['video_id', 'file_name']).label.nunique().max() == 1,
            'Conflicting duplicate labels; do not silently discard these')
    before = len(df)
    df = df.drop_duplicates().sort_values(['video_id', 'file_name']).reset_index(drop=True)
    require(not df.file_name.duplicated().any(), 'Filename assigned to multiple videos')
    require(len(df) == spec['rows'] and df.video_id.nunique() == spec['videos'], 'Unexpected dataset size')
    require(df.label.value_counts().sort_index().to_dict() == spec['counts'], 'Unexpected class counts')
    require(table_hash(df, ['file_name', 'video_id', 'label']) == spec['expected_hash'],
            'Metadata differs from the audited CNN label files. Investigate before training.')
    if task == 'severity':
        require(df.groupby('video_id').label.nunique().max() == 1, 'Mixed severity labels')
    print(task, 'removed exact duplicate rows:', before-len(df))
    return df

def extract_archive(path, task):
    digest = file_hash(path)
    root = DATA_ROOT / task / ('extracted_' + digest[:16])
    marker = root / 'complete.json'
    if marker.exists():
        require(json.loads(marker.read_text())['sha256'] == digest, 'Extraction checksum mismatch')
        return root
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as z:
        names = set()
        for info in z.infolist():
            dest = (root / info.filename).resolve()
            require(dest == root.resolve() or root.resolve() in dest.parents, 'Unsafe ZIP path')
            require((info.external_attr >> 16) & 0o170000 != 0o120000, 'ZIP symlink rejected')
            if not info.is_dir():
                require(dest not in names, 'Duplicate ZIP member')
                names.add(dest)
        z.extractall(root)
    save_json(marker, {'sha256':digest})
    return root

def attach_images(df, root):
    from PIL import Image
    index = defaultdict(list)
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}:
            index[(p.parent.name.strip(), p.name.strip())].append(p)
    df = df.copy()
    paths, hashes = [], []
    for row in df.itertuples():
        matches = index[(row.video_id, row.file_name)]
        require(len(matches) == 1, f'Missing/ambiguous image: {row.video_id}/{row.file_name}')
        p = matches[0]
        with Image.open(p) as im:
            im.convert('RGB').load()  # Fail before training on unreadable images.
        paths.append(str(p)); hashes.append(file_hash(p))
    df['image_path'], df['image_sha256'] = paths, hashes
    print('Unreferenced archive images:', sum(map(len,index.values()))-len(df))
    return df

def assign_splits(df, seed):
    groups = df.groupby('video_id', sort=True).label.agg(lambda x: ','.join(map(str,sorted(set(x)))))
    train_ids, rest_ids = train_test_split(groups.index.to_numpy(), test_size=.30,
                                          stratify=groups.to_numpy(), random_state=seed)
    val_ids, test_ids = train_test_split(rest_ids, test_size=.50,
                                        stratify=groups.loc[rest_ids].to_numpy(), random_state=seed)
    mapping = {v:s for s,ids in [('train',train_ids),('val',val_ids),('test',test_ids)] for v in ids}
    out = df.copy(); out['split'] = out.video_id.map(mapping)
    require(out.groupby('video_id').split.nunique().eq(1).all(), 'Patient leakage')
    for split in ['train','val','test']:
        require(set(out.loc[out.split==split,'label']) == set(out.label), f'Missing class in {split}')
    if 'image_sha256' in out:
        require(out.groupby('image_sha256').split.nunique().max() == 1,
                'Identical image bytes cross splits. Resolve source identities before training.')
    return out

def prepare_task(task):
    meta_path, zip_path = fetch(task,'metadata'), fetch(task,'zip')
    df = canonical_metadata(meta_path, task)
    df = attach_images(df, extract_archive(zip_path, task))
    df = assign_splits(df, CONFIG['seed'])
    directory = OUTPUT_ROOT / task
    directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / 'split_manifest.csv'
    cols = ['file_name','video_id','label','image_sha256','split']
    if manifest.exists():
        require(table_hash(pd.read_csv(manifest),cols)==table_hash(df,cols),
                'Dataset or split changed for this RUN_NAME. Use a new RUN_NAME.')
    df.to_csv(manifest, index=False)
    save_json(directory/'data_sources.json', {**SOURCES[task], 'metadata_sha256':file_hash(meta_path),
                                              'archive_sha256':file_hash(zip_path)})
    print(task, df.groupby('split').agg(frames=('file_name','size'), videos=('video_id','nunique')))
    print(pd.crosstab(df.split,df.label))
    return df



4. Run data checks (no GPU needed)
Exact image hashing detects identical files, not re-encoded duplicates or patient
aliases. The supplied video-to-patient identity remains the grouping assumption.


In [ ]:
DATA = {task:prepare_task(task) for task in TASKS}


5. Model wrapper and configurable fine-tuning
Uses the original feature-L2-normalization plus a new linear head.
Distilled DeiT uses its pretrained distilled backbone; no teacher/distillation
loss is trained on the cataract dataset. Printed trainable names verify scope.


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm, sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score

REGISTRY = dict(zip(
    ['ViT-B/16','ViT-B/32','DeiT-Tiny','DeiT-Small','DeiT-Base','DeiT-Small-Distilled','Swin-Tiny','Swin-Small','Swin-Base'],
    ['vit_base_patch16_224','vit_base_patch32_224','deit_tiny_patch16_224','deit_small_patch16_224',
     'deit_base_patch16_224','deit_small_distilled_patch16_224','swin_tiny_patch4_window7_224',
     'swin_small_patch4_window7_224','swin_base_patch4_window7_224']))

class TransformerClassifier(nn.Module):
    def __init__(self, name, classes, cfg, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(REGISTRY[name], pretrained=pretrained, num_classes=0)
        self.binary = len(classes)==2
        self.classifier = nn.Linear(self.backbone.num_features, 1 if self.binary else len(classes))
        self.l2 = cfg['l2_features']
        self.mode, self.depth = TUNE_OVERRIDES.get(name,(cfg['tune_mode'],cfg['tune_depth']))
        self.unfrozen_modules = []
        if self.mode != 'all':
            self.backbone.requires_grad_(False)
            if self.mode == 'last_blocks':
                require(hasattr(self.backbone,'blocks'), 'last_blocks requires ViT/DeiT')
                require(1<=self.depth<=len(self.backbone.blocks), 'Invalid block depth')
                self.unfrozen_modules = list(self.backbone.blocks[-self.depth:])
            elif self.mode == 'last_stages':
                require(hasattr(self.backbone,'layers'), 'last_stages requires Swin')
                require(1<=self.depth<=len(self.backbone.layers), 'Invalid stage depth')
                self.unfrozen_modules = list(self.backbone.layers[-self.depth:])
            else:
                require(self.mode=='head', 'Unknown fine-tuning mode')
            if self.mode != 'head':
                for attr in ['norm','fc_norm']:
                    if isinstance(getattr(self.backbone,attr,None), nn.Module):
                        self.unfrozen_modules.append(getattr(self.backbone,attr))
            for module in self.unfrozen_modules:
                module.requires_grad_(True)

    def train(self, mode=True):
        super().train(mode)
        if mode and self.mode != 'all':
            self.backbone.eval()
            for module in self.unfrozen_modules:
                module.train(True)
        return self

    def embeddings(self, x):
        features = self.backbone(x)
        require(isinstance(features,torch.Tensor) and features.ndim==2, 'Expected pooled embeddings')
        return F.normalize(features,p=2,dim=1) if self.l2 else features

    def forward(self,x):
        logits = self.classifier(self.embeddings(x))
        return logits.squeeze(-1) if self.binary else logits



6. Loaders and metrics
Training sampler is weighted by training frame counts only. Validation/test use
all frames exactly once. Probabilities are always in the saved class order.
Frame metrics are primary; no arbitrary whole-video label is assigned to mixed
poor-dilation videos. Frame predictions retain patient identifiers for analysis.


In [ ]:
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

class FrameDataset(Dataset):
    def __init__(self, df, augment):
        self.df = df.reset_index(drop=True)
        ops = [transforms.Resize((224,224))]
        if augment:
            ops += [transforms.RandomHorizontalFlip(.5), transforms.RandomVerticalFlip(.3),
                    transforms.RandomRotation(15), transforms.RandomAffine(0,translate=(.1,.1))]
        self.transform=transforms.Compose(ops+[transforms.ToTensor(),
            transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    def __len__(self): return len(self.df)
    def __getitem__(self,index):
        from PIL import Image
        row=self.df.iloc[index]
        with Image.open(row.image_path) as im: image=self.transform(im.convert('RGB'))
        return image,int(row.label)

def make_loaders(df,cfg):
    loaders={}
    for split in ['train','val','test']:
        ds=FrameDataset(df[df.split==split],split=='train')
        sampler=None
        if split=='train':
            counts=ds.df.label.value_counts()
            weights=ds.df.label.map(lambda v:1/counts[v]).to_numpy()
            sampler=WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),len(ds),replacement=True)
        loaders[split]=DataLoader(ds,batch_size=cfg['batch_size'],sampler=sampler,
                                  shuffle=False,num_workers=cfg['workers'],pin_memory=True,drop_last=False)
    return loaders

def epoch_pass(model,loader,device,optimizer=None,scaler=None):
    training=optimizer is not None
    model.train(training)
    criterion=nn.BCEWithLogitsLoss() if model.binary else nn.CrossEntropyLoss()
    total,count=0.,0
    labels_list,probs_list=[],[]
    with torch.set_grad_enabled(training):
        for images,labels in loader:
            images=images.to(device); labels=labels.to(device)
            target=labels.float() if model.binary else labels.long()
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,enabled=training and device.type=='cuda'):
                logits=model(images); loss=criterion(logits,target)
            require(torch.isfinite(loss).item(),'Non-finite loss')
            if training:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total+=loss.item()*len(labels); count+=len(labels)
            if model.binary:
                p=logits.detach().float().sigmoid(); probabilities=torch.stack([1-p,p],dim=1)
            else: probabilities=logits.detach().float().softmax(dim=1)
            probs_list.append(probabilities.cpu().numpy()); labels_list.append(labels.cpu().numpy())
    return total/count,np.concatenate(labels_list),np.concatenate(probs_list)

def metrics(y,p):
    pred=p.argmax(axis=1); classes=list(range(p.shape[1]))
    out={'accuracy':float(accuracy_score(y,pred)),
         'macro_f1':float(f1_score(y,pred,labels=classes,average='macro',zero_division=0)),
         'weighted_f1':float(f1_score(y,pred,labels=classes,average='weighted',zero_division=0)),
         'confusion_matrix':confusion_matrix(y,pred,labels=classes).tolist(),'n_frames':len(y)}
    if p.shape[1]==2:
        out.update(poor_precision=float(precision_score(y,pred,zero_division=0)),
                   poor_recall=float(recall_score(y,pred,zero_division=0)),
                   specificity=float(recall_score(y,pred,pos_label=0,zero_division=0)),
                   roc_auc=float(roc_auc_score(y,p[:,1])))
    return out



7. Training, guarded resume, best-checkpoint evaluation and durable artifacts
Metrics are fractions in [0,1], never formatted percentage strings. Completed
runs are skipped only when their protocol and required output files match.
Resume restores epoch, weights, optimizer, scheduler, scaler and RNG state.
Load checkpoints only from your own trusted run outputs.


In [ ]:
def save_checkpoint(path,obj):
    temp=path.with_suffix('.tmp'); torch.save(obj,temp); temp.replace(path)

def rng_state():
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if torch.cuda.is_available(): torch.cuda.set_rng_state(state['cuda'][0], device=0)

def train_run(task,name,df):
    seed_all(CONFIG['seed'])
    require(torch.cuda.is_available(),'Enable a Kaggle GPU before training')
    device=torch.device('cuda:0')
    folder=OUTPUT_ROOT/task/name.replace('/','_'); folder.mkdir(parents=True,exist_ok=True)
    mode,depth=TUNE_OVERRIDES.get(name,(CONFIG['tune_mode'],CONFIG['tune_depth']))
    classes=SOURCES[task]['classes']
    protocol=dict(task=task,model=name,timm_name=REGISTRY[name],classes=classes,config=CONFIG,
                  tuning=[mode,depth],split_hash=table_hash(df,['file_name','video_id','label','image_sha256','split']),
                  versions=dict(torch=torch.__version__,timm=timm.__version__,sklearn=sklearn.__version__,
                                numpy=np.__version__,python=platform.python_version()))
    protocol_path=folder/'protocol.json'
    if protocol_path.exists():
        require(json.loads(protocol_path.read_text())==protocol, 'Run protocol changed. Use a new RUN_NAME.')
    else: save_json(protocol_path,protocol)
    required=['best.pth','history.csv','val_predictions.csv','test_predictions.csv','metrics.json']
    if (folder/'complete.json').exists():
        require(all((folder/p).is_file() and (folder/p).stat().st_size for p in required),'Incomplete completed run')
        print('Skipping completed',task,name); return
    last=folder/'last.pth'; best_path=folder/'best.pth'
    model=TransformerClassifier(name,classes,CONFIG,pretrained=not last.exists()).to(device)
    loaders=make_loaders(df,CONFIG)
    backbone=[p for p in model.backbone.parameters() if p.requires_grad]
    groups=[{'params':model.classifier.parameters(),'lr':CONFIG['head_lr'],'name':'head'}]
    if backbone: groups.append({'params':backbone,'lr':CONFIG['backbone_lr'],'name':'backbone'})
    optimizer=torch.optim.AdamW(groups,weight_decay=CONFIG['weight_decay'])
    print('Initial learning rates:', {g['name']:g['lr'] for g in optimizer.param_groups})
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CONFIG['epochs'])
    scaler=torch.amp.GradScaler('cuda')
    print(task,name,'trainable/total parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad),
          '/',sum(p.numel() for p in model.parameters()))
    (folder/'trainable_parameters.txt').write_text('\n'.join(n for n,p in model.named_parameters() if p.requires_grad))
    save_json(folder/'pretrained_config.json',model.backbone.pretrained_cfg)
    start,best,bad,history=0,float('inf'),0,[]
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        require(state['protocol']==protocol,'Checkpoint protocol mismatch')
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
        start,best,bad,history=state['epoch']+1,state['best_loss'],state['bad_epochs'],state['history']
        restore_rng(state['rng'])
        require(best_path.exists(),'Resume is missing the best checkpoint')
        del state
    for epoch in range(start,CONFIG['epochs']):
        if bad>=CONFIG['patience']: break
        epoch_lrs={g['name']+'_lr':float(g['lr']) for g in optimizer.param_groups}
        train_loss,yt,pt=epoch_pass(model,loaders['train'],device,optimizer,scaler)
        val_loss,yv,pv=epoch_pass(model,loaders['val'],device)
        history.append(dict(epoch=epoch+1,train_loss=train_loss,val_loss=val_loss,
                            train_accuracy=float(accuracy_score(yt,pt.argmax(1))),
                            val_accuracy=float(accuracy_score(yv,pv.argmax(1))),**epoch_lrs))
        if val_loss<best:
            best,bad=val_loss,0
            save_checkpoint(best_path,dict(model={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                                          epoch=epoch+1,val_loss=val_loss,protocol=protocol))
        else: bad+=1
        scheduler.step()
        save_checkpoint(last,dict(model={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                                   optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
                                   scaler=scaler.state_dict(),epoch=epoch,best_loss=best,bad_epochs=bad,
                                   history=history,rng=rng_state(),protocol=protocol))
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        print(task,name,history[-1],flush=True)
    checkpoint=torch.load(best_path,map_location='cpu',weights_only=False)
    model.load_state_dict(checkpoint['model'])
    result=dict(task=task,model=name,best_epoch=checkpoint['epoch'],best_val_loss=checkpoint['val_loss'])
    for split in ['val','test']:
        loss,y,p=epoch_pass(model,loaders[split],device)
        result[split]=dict(loss=loss,**metrics(y,p))
        predictions=loaders[split].dataset.df[['file_name','video_id','label']].copy()
        require(np.array_equal(predictions.label.to_numpy(),y),'Prediction order mismatch')
        predictions['prediction']=p.argmax(1)
        for c in range(len(classes)): predictions[f'prob_{c}']=p[:,c]
        predictions.to_csv(folder/f'{split}_predictions.csv',index=False)
    save_json(folder/'metrics.json',result)
    save_json(folder/'complete.json',{'best_epoch':checkpoint['epoch'],'split_hash':protocol['split_hash']})
    print(json.dumps(result,indent=2))
    del model,optimizer,scheduler,scaler,checkpoint,loaders
    gc.collect(); torch.cuda.empty_cache()



8. Train on both T4 GPUs

Select Kaggle GPU T4 x2. Two isolated processes train different models concurrently, one per GPU. The next model starts when a GPU becomes free. Each model keeps its full batch size of 32 and its own RNG, optimizer and checkpoints. Data preparation runs once before workers start.

The parent updates summaries; workers write separate model folders. Per-model console logs are saved in `logs/`. Interrupting this cell stops both active workers; their last completed epoch remains resumable. The embedded training worker uses section 2 settings. Changes to model/training definitions also require updating the embedded worker source.


In [ ]:
def write_task_summary(task):
    directory = OUTPUT_ROOT / task
    directory.mkdir(parents=True, exist_ok=True)
    rows, status = [], []
    for name in MODEL_NAMES:
        folder = directory / name.replace('/', '_')
        complete = (folder/'complete.json').exists()
        status.append(dict(model=name, status='complete' if complete else
                           'partial' if (folder/'last.pth').exists() else 'not_started'))
        if not complete: continue
        r = json.loads((folder/'metrics.json').read_text())
        row = {k:r[k] for k in ['task','model','best_epoch','best_val_loss']}
        row['checkpoint'] = str((folder/'best.pth').relative_to(directory))
        for split in ['val','test']:
            row.update({split+'_'+k:v for k,v in r[split].items() if k!='confusion_matrix'})
        rows.append(row)
    save_json(directory/'run_status.json', status)
    (directory/'README.txt').write_text(
        'Each model directory contains best.pth (weights selected by validation loss), '
        'last.pth (full training state for resume), history.csv, metrics.json, '
        'val_predictions.csv, test_predictions.csv, protocol.json, pretrained_config.json, '
        'trainable_parameters.txt and complete.json.\n'
        'summary.csv ranks completed models by validation macro F1. '
        'best_model.json points to the top completed model; selection is provisional '
        'until all requested models finish. Test metrics never select the winner.\n'
        'split_manifest.csv and data_sources.json identify the data and patient split.\n'
        'Download this complete task folder to preserve both results and resumable checkpoints.\n'
        'To restore a model, construct TransformerClassifier using the protocol model, '
        'classes and config with pretrained=False, then load checkpoint["model"]. '
        'Only load trusted checkpoints.\n')
    if not rows: return None
    summary = pd.DataFrame(rows).sort_values(['val_macro_f1','model'],ascending=[False,True])
    summary.to_csv(directory/'summary.csv',index=False)
    winner = summary.iloc[0]
    save_json(directory/'best_model.json', dict(model=str(winner['model']),
        checkpoint=str(winner['checkpoint']), selection_metric='val_macro_f1',
        selection_value=float(winner['val_macro_f1']),
        completed_models=len(rows), requested_models=len(MODEL_NAMES),
        selection_final=len(rows)==len(MODEL_NAMES)))
    return summary


WORKER_SOURCE = "import sys\nfrom pathlib import Path\nimport os, json, hashlib, zipfile, random, copy, gc, platform\nfrom collections import defaultdict\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()\nDATA_ROOT = ROOT / 'transformer_data_v2'\nRUN_NAME = 'transformers_severity_v2_two_lr'\nOUTPUT_ROOT = ROOT / 'outputs' / RUN_NAME\nRUN_TRAINING = False\nTASKS = ['severity']\nMODEL_NAMES = ['ViT-B/16', 'ViT-B/32', 'DeiT-Tiny', 'DeiT-Small',\n               'DeiT-Base', 'DeiT-Small-Distilled', 'Swin-Tiny', 'Swin-Small', 'Swin-Base']\nCONFIG = dict(seed=42, epochs=30, batch_size=32, workers=2, patience=5,\n              backbone_lr=1e-5, head_lr=1e-4, weight_decay=0.01,\n              tune_mode='all', tune_depth=4, l2_features=True)\nTUNE_OVERRIDES = {}\nSOURCES = {\n    'severity': dict(zip_id='117lqrb69UR41ZeQXtTX895dE3sF8iTmo',\n                     metadata_id='1YSX2UP_paaqiCQ9W1mzVKub7GvJDL7i7', ext='xlsx',\n                     classes=['low nuclear density', 'dense', 'mature', 'brunescent'],\n                     rows=2541, videos=645, counts={0:795, 1:1047, 2:614, 3:85},\n                     expected_hash='1bcc1c61c024845835cabc7ad020f6d1128a2a3ff929e03bf0d8fd53ee4b5e50'),\n    'poor_dilation': dict(zip_id='1hpCGfGdu-wjB2oi0xFNzb-pF4-mWjbWi',\n                          metadata_id='1rHw_AlqBx_wdmsKtlJWlzt8-Hs4ZdSmF', ext='csv',\n                          classes=['normal', 'poor dilation'], rows=3081, videos=700,\n                          counts={0:2541, 1:540},\n                          expected_hash='dd8462f5408f9637e63ab7c8f70325f23a90db6a58da5080f81d1f6033150a4f')}\nLOCAL_FILES = {t: {'zip': None, 'metadata': None} for t in SOURCES}\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b''):\n            h.update(chunk)\n    return h.hexdigest()\n\ndef table_hash(df, columns):\n    data = df[columns].sort_values(['video_id', 'file_name']).reset_index(drop=True)\n    return hashlib.sha256(data.to_csv(index=False, lineterminator='\\n').encode()).hexdigest()\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + '.tmp')\n    temp.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False))\n    temp.replace(path)\n\ndef fetch(task, kind):\n    import gdown, requests\n    spec = SOURCES[task]\n    override = LOCAL_FILES[task][kind]\n    if override:\n        p = Path(override)\n        require(p.is_file(), f'Missing local file: {p}')\n        return p\n    p = DATA_ROOT / task / ('images.zip' if kind == 'zip' else 'labels.' + spec['ext'])\n    p.parent.mkdir(parents=True, exist_ok=True)\n    if p.is_file():\n        return p\n    part = p.with_suffix(p.suffix + '.part')\n    if kind == 'metadata' and spec['ext'] == 'xlsx':\n        response = requests.get('https://docs.google.com/spreadsheets/d/' +\n                                spec['metadata_id'] + '/export?format=xlsx', timeout=180)\n        response.raise_for_status()\n        part.write_bytes(response.content)\n    else:\n        result = gdown.download(id=spec['zip_id'] if kind == 'zip' else spec['metadata_id'],\n                                output=str(part), quiet=False)\n        require(result is not None, 'Drive download failed. Check sharing or use LOCAL_FILES.')\n    require(part.is_file() and part.stat().st_size > 0, f'Empty download: {p}')\n    if kind == 'zip' or spec['ext'] == 'xlsx':\n        require(zipfile.is_zipfile(part), f'Invalid ZIP/XLSX: {p}')\n    part.replace(p)\n    return p\n\ndef canonical_metadata(path, task):\n    spec = SOURCES[task]\n    raw = pd.read_excel(path, sheet_name='Dataset_Severity_V2') if spec['ext'] == 'xlsx' else pd.read_csv(path)\n    raw.columns = raw.columns.astype(str).str.strip()\n    columns = ['filename', 'videoname'] + spec['classes']\n    require(set(columns) <= set(raw.columns), 'Missing metadata columns')\n    require(not raw[columns].isna().any().any(), 'Missing metadata values')\n    labels = raw[spec['classes']]\n    require(labels.isin([0, 1]).all().all() and labels.sum(axis=1).eq(1).all(), 'Invalid one-hot labels')\n    df = pd.DataFrame(dict(file_name=raw.filename.astype(str).str.strip(),\n                           video_id=raw.videoname.astype(str).str.strip(),\n                           label=labels.to_numpy().argmax(axis=1)))\n    require(df.file_name.ne('').all() and df.video_id.ne('').all(), 'Empty identifiers')\n    require(df.groupby(['video_id', 'file_name']).label.nunique().max() == 1,\n            'Conflicting duplicate labels; do not silently discard these')\n    before = len(df)\n    df = df.drop_duplicates().sort_values(['video_id', 'file_name']).reset_index(drop=True)\n    require(not df.file_name.duplicated().any(), 'Filename assigned to multiple videos')\n    require(len(df) == spec['rows'] and df.video_id.nunique() == spec['videos'], 'Unexpected dataset size')\n    require(df.label.value_counts().sort_index().to_dict() == spec['counts'], 'Unexpected class counts')\n    require(table_hash(df, ['file_name', 'video_id', 'label']) == spec['expected_hash'],\n            'Metadata differs from the audited CNN label files. Investigate before training.')\n    if task == 'severity':\n        require(df.groupby('video_id').label.nunique().max() == 1, 'Mixed severity labels')\n    print(task, 'removed exact duplicate rows:', before-len(df))\n    return df\n\ndef extract_archive(path, task):\n    digest = file_hash(path)\n    root = DATA_ROOT / task / ('extracted_' + digest[:16])\n    marker = root / 'complete.json'\n    if marker.exists():\n        require(json.loads(marker.read_text())['sha256'] == digest, 'Extraction checksum mismatch')\n        return root\n    root.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(path) as z:\n        names = set()\n        for info in z.infolist():\n            dest = (root / info.filename).resolve()\n            require(dest == root.resolve() or root.resolve() in dest.parents, 'Unsafe ZIP path')\n            require((info.external_attr >> 16) & 0o170000 != 0o120000, 'ZIP symlink rejected')\n            if not info.is_dir():\n                require(dest not in names, 'Duplicate ZIP member')\n                names.add(dest)\n        z.extractall(root)\n    save_json(marker, {'sha256':digest})\n    return root\n\ndef attach_images(df, root):\n    from PIL import Image\n    index = defaultdict(list)\n    for p in sorted(root.rglob('*')):\n        if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}:\n            index[(p.parent.name.strip(), p.name.strip())].append(p)\n    df = df.copy()\n    paths, hashes = [], []\n    for row in df.itertuples():\n        matches = index[(row.video_id, row.file_name)]\n        require(len(matches) == 1, f'Missing/ambiguous image: {row.video_id}/{row.file_name}')\n        p = matches[0]\n        with Image.open(p) as im:\n            im.convert('RGB').load()  # Fail before training on unreadable images.\n        paths.append(str(p)); hashes.append(file_hash(p))\n    df['image_path'], df['image_sha256'] = paths, hashes\n    print('Unreferenced archive images:', sum(map(len,index.values()))-len(df))\n    return df\n\ndef assign_splits(df, seed):\n    groups = df.groupby('video_id', sort=True).label.agg(lambda x: ','.join(map(str,sorted(set(x)))))\n    train_ids, rest_ids = train_test_split(groups.index.to_numpy(), test_size=.30,\n                                          stratify=groups.to_numpy(), random_state=seed)\n    val_ids, test_ids = train_test_split(rest_ids, test_size=.50,\n                                        stratify=groups.loc[rest_ids].to_numpy(), random_state=seed)\n    mapping = {v:s for s,ids in [('train',train_ids),('val',val_ids),('test',test_ids)] for v in ids}\n    out = df.copy(); out['split'] = out.video_id.map(mapping)\n    require(out.groupby('video_id').split.nunique().eq(1).all(), 'Patient leakage')\n    for split in ['train','val','test']:\n        require(set(out.loc[out.split==split,'label']) == set(out.label), f'Missing class in {split}')\n    if 'image_sha256' in out:\n        require(out.groupby('image_sha256').split.nunique().max() == 1,\n                'Identical image bytes cross splits. Resolve source identities before training.')\n    return out\n\ndef prepare_task(task):\n    meta_path, zip_path = fetch(task,'metadata'), fetch(task,'zip')\n    df = canonical_metadata(meta_path, task)\n    df = attach_images(df, extract_archive(zip_path, task))\n    df = assign_splits(df, CONFIG['seed'])\n    directory = OUTPUT_ROOT / task\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest = directory / 'split_manifest.csv'\n    cols = ['file_name','video_id','label','image_sha256','split']\n    if manifest.exists():\n        require(table_hash(pd.read_csv(manifest),cols)==table_hash(df,cols),\n                'Dataset or split changed for this RUN_NAME. Use a new RUN_NAME.')\n    df.to_csv(manifest, index=False)\n    save_json(directory/'data_sources.json', {**SOURCES[task], 'metadata_sha256':file_hash(meta_path),\n                                              'archive_sha256':file_hash(zip_path)})\n    print(task, df.groupby('split').agg(frames=('file_name','size'), videos=('video_id','nunique')))\n    print(pd.crosstab(df.split,df.label))\n    return df\n\n\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\nfrom torchvision import transforms\nimport timm, sklearn\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score\n\nREGISTRY = dict(zip(\n    ['ViT-B/16','ViT-B/32','DeiT-Tiny','DeiT-Small','DeiT-Base','DeiT-Small-Distilled','Swin-Tiny','Swin-Small','Swin-Base'],\n    ['vit_base_patch16_224','vit_base_patch32_224','deit_tiny_patch16_224','deit_small_patch16_224',\n     'deit_base_patch16_224','deit_small_distilled_patch16_224','swin_tiny_patch4_window7_224',\n     'swin_small_patch4_window7_224','swin_base_patch4_window7_224']))\n\nclass TransformerClassifier(nn.Module):\n    def __init__(self, name, classes, cfg, pretrained=True):\n        super().__init__()\n        self.backbone = timm.create_model(REGISTRY[name], pretrained=pretrained, num_classes=0)\n        self.binary = len(classes)==2\n        self.classifier = nn.Linear(self.backbone.num_features, 1 if self.binary else len(classes))\n        self.l2 = cfg['l2_features']\n        self.mode, self.depth = TUNE_OVERRIDES.get(name,(cfg['tune_mode'],cfg['tune_depth']))\n        self.unfrozen_modules = []\n        if self.mode != 'all':\n            self.backbone.requires_grad_(False)\n            if self.mode == 'last_blocks':\n                require(hasattr(self.backbone,'blocks'), 'last_blocks requires ViT/DeiT')\n                require(1<=self.depth<=len(self.backbone.blocks), 'Invalid block depth')\n                self.unfrozen_modules = list(self.backbone.blocks[-self.depth:])\n            elif self.mode == 'last_stages':\n                require(hasattr(self.backbone,'layers'), 'last_stages requires Swin')\n                require(1<=self.depth<=len(self.backbone.layers), 'Invalid stage depth')\n                self.unfrozen_modules = list(self.backbone.layers[-self.depth:])\n            else:\n                require(self.mode=='head', 'Unknown fine-tuning mode')\n            if self.mode != 'head':\n                for attr in ['norm','fc_norm']:\n                    if isinstance(getattr(self.backbone,attr,None), nn.Module):\n                        self.unfrozen_modules.append(getattr(self.backbone,attr))\n            for module in self.unfrozen_modules:\n                module.requires_grad_(True)\n\n    def train(self, mode=True):\n        super().train(mode)\n        if mode and self.mode != 'all':\n            self.backbone.eval()\n            for module in self.unfrozen_modules:\n                module.train(True)\n        return self\n\n    def embeddings(self, x):\n        features = self.backbone(x)\n        require(isinstance(features,torch.Tensor) and features.ndim==2, 'Expected pooled embeddings')\n        return F.normalize(features,p=2,dim=1) if self.l2 else features\n\n    def forward(self,x):\n        logits = self.classifier(self.embeddings(x))\n        return logits.squeeze(-1) if self.binary else logits\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cudnn.benchmark=False\n\nclass FrameDataset(Dataset):\n    def __init__(self, df, augment):\n        self.df = df.reset_index(drop=True)\n        ops = [transforms.Resize((224,224))]\n        if augment:\n            ops += [transforms.RandomHorizontalFlip(.5), transforms.RandomVerticalFlip(.3),\n                    transforms.RandomRotation(15), transforms.RandomAffine(0,translate=(.1,.1))]\n        self.transform=transforms.Compose(ops+[transforms.ToTensor(),\n            transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\n    def __len__(self): return len(self.df)\n    def __getitem__(self,index):\n        from PIL import Image\n        row=self.df.iloc[index]\n        with Image.open(row.image_path) as im: image=self.transform(im.convert('RGB'))\n        return image,int(row.label)\n\ndef make_loaders(df,cfg):\n    loaders={}\n    for split in ['train','val','test']:\n        ds=FrameDataset(df[df.split==split],split=='train')\n        sampler=None\n        if split=='train':\n            counts=ds.df.label.value_counts()\n            weights=ds.df.label.map(lambda v:1/counts[v]).to_numpy()\n            sampler=WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),len(ds),replacement=True)\n        loaders[split]=DataLoader(ds,batch_size=cfg['batch_size'],sampler=sampler,\n                                  shuffle=False,num_workers=cfg['workers'],pin_memory=True,drop_last=False)\n    return loaders\n\ndef epoch_pass(model,loader,device,optimizer=None,scaler=None):\n    training=optimizer is not None\n    model.train(training)\n    criterion=nn.BCEWithLogitsLoss() if model.binary else nn.CrossEntropyLoss()\n    total,count=0.,0\n    labels_list,probs_list=[],[]\n    with torch.set_grad_enabled(training):\n        for images,labels in loader:\n            images=images.to(device); labels=labels.to(device)\n            target=labels.float() if model.binary else labels.long()\n            if training: optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=training and device.type=='cuda'):\n                logits=model(images); loss=criterion(logits,target)\n            require(torch.isfinite(loss).item(),'Non-finite loss')\n            if training:\n                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()\n            total+=loss.item()*len(labels); count+=len(labels)\n            if model.binary:\n                p=logits.detach().float().sigmoid(); probabilities=torch.stack([1-p,p],dim=1)\n            else: probabilities=logits.detach().float().softmax(dim=1)\n            probs_list.append(probabilities.cpu().numpy()); labels_list.append(labels.cpu().numpy())\n    return total/count,np.concatenate(labels_list),np.concatenate(probs_list)\n\ndef metrics(y,p):\n    pred=p.argmax(axis=1); classes=list(range(p.shape[1]))\n    out={'accuracy':float(accuracy_score(y,pred)),\n         'macro_f1':float(f1_score(y,pred,labels=classes,average='macro',zero_division=0)),\n         'weighted_f1':float(f1_score(y,pred,labels=classes,average='weighted',zero_division=0)),\n         'confusion_matrix':confusion_matrix(y,pred,labels=classes).tolist(),'n_frames':len(y)}\n    if p.shape[1]==2:\n        out.update(poor_precision=float(precision_score(y,pred,zero_division=0)),\n                   poor_recall=float(recall_score(y,pred,zero_division=0)),\n                   specificity=float(recall_score(y,pred,pos_label=0,zero_division=0)),\n                   roc_auc=float(roc_auc_score(y,p[:,1])))\n    return out\n\n\ndef save_checkpoint(path,obj):\n    temp=path.with_suffix('.tmp'); torch.save(obj,temp); temp.replace(path)\n\ndef rng_state():\n    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\ndef restore_rng(state):\n    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])\n    if torch.cuda.is_available(): torch.cuda.set_rng_state(state['cuda'][0], device=0)\n\ndef train_run(task,name,df):\n    seed_all(CONFIG['seed'])\n    require(torch.cuda.is_available(),'Enable a Kaggle GPU before training')\n    device=torch.device('cuda:0')\n    folder=OUTPUT_ROOT/task/name.replace('/','_'); folder.mkdir(parents=True,exist_ok=True)\n    mode,depth=TUNE_OVERRIDES.get(name,(CONFIG['tune_mode'],CONFIG['tune_depth']))\n    classes=SOURCES[task]['classes']\n    protocol=dict(task=task,model=name,timm_name=REGISTRY[name],classes=classes,config=CONFIG,\n                  tuning=[mode,depth],split_hash=table_hash(df,['file_name','video_id','label','image_sha256','split']),\n                  versions=dict(torch=torch.__version__,timm=timm.__version__,sklearn=sklearn.__version__,\n                                numpy=np.__version__,python=platform.python_version()))\n    protocol_path=folder/'protocol.json'\n    if protocol_path.exists():\n        require(json.loads(protocol_path.read_text())==protocol, 'Run protocol changed. Use a new RUN_NAME.')\n    else: save_json(protocol_path,protocol)\n    required=['best.pth','history.csv','val_predictions.csv','test_predictions.csv','metrics.json']\n    if (folder/'complete.json').exists():\n        require(all((folder/p).is_file() and (folder/p).stat().st_size for p in required),'Incomplete completed run')\n        print('Skipping completed',task,name); return\n    last=folder/'last.pth'; best_path=folder/'best.pth'\n    model=TransformerClassifier(name,classes,CONFIG,pretrained=not last.exists()).to(device)\n    loaders=make_loaders(df,CONFIG)\n    backbone=[p for p in model.backbone.parameters() if p.requires_grad]\n    groups=[{'params':model.classifier.parameters(),'lr':CONFIG['head_lr'],'name':'head'}]\n    if backbone: groups.append({'params':backbone,'lr':CONFIG['backbone_lr'],'name':'backbone'})\n    optimizer=torch.optim.AdamW(groups,weight_decay=CONFIG['weight_decay'])\n    print('Initial learning rates:', {g['name']:g['lr'] for g in optimizer.param_groups})\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CONFIG['epochs'])\n    scaler=torch.amp.GradScaler('cuda')\n    print(task,name,'trainable/total parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad),\n          '/',sum(p.numel() for p in model.parameters()))\n    (folder/'trainable_parameters.txt').write_text('\\n'.join(n for n,p in model.named_parameters() if p.requires_grad))\n    save_json(folder/'pretrained_config.json',model.backbone.pretrained_cfg)\n    start,best,bad,history=0,float('inf'),0,[]\n    if last.exists():\n        state=torch.load(last,map_location='cpu',weights_only=False)\n        require(state['protocol']==protocol,'Checkpoint protocol mismatch')\n        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])\n        scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])\n        start,best,bad,history=state['epoch']+1,state['best_loss'],state['bad_epochs'],state['history']\n        restore_rng(state['rng'])\n        require(best_path.exists(),'Resume is missing the best checkpoint')\n        del state\n    for epoch in range(start,CONFIG['epochs']):\n        if bad>=CONFIG['patience']: break\n        epoch_lrs={g['name']+'_lr':float(g['lr']) for g in optimizer.param_groups}\n        train_loss,yt,pt=epoch_pass(model,loaders['train'],device,optimizer,scaler)\n        val_loss,yv,pv=epoch_pass(model,loaders['val'],device)\n        history.append(dict(epoch=epoch+1,train_loss=train_loss,val_loss=val_loss,\n                            train_accuracy=float(accuracy_score(yt,pt.argmax(1))),\n                            val_accuracy=float(accuracy_score(yv,pv.argmax(1))),**epoch_lrs))\n        if val_loss<best:\n            best,bad=val_loss,0\n            save_checkpoint(best_path,dict(model={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},\n                                          epoch=epoch+1,val_loss=val_loss,protocol=protocol))\n        else: bad+=1\n        scheduler.step()\n        save_checkpoint(last,dict(model={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},\n                                   optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),\n                                   scaler=scaler.state_dict(),epoch=epoch,best_loss=best,bad_epochs=bad,\n                                   history=history,rng=rng_state(),protocol=protocol))\n        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)\n        print(task,name,history[-1],flush=True)\n    checkpoint=torch.load(best_path,map_location='cpu',weights_only=False)\n    model.load_state_dict(checkpoint['model'])\n    result=dict(task=task,model=name,best_epoch=checkpoint['epoch'],best_val_loss=checkpoint['val_loss'])\n    for split in ['val','test']:\n        loss,y,p=epoch_pass(model,loaders[split],device)\n        result[split]=dict(loss=loss,**metrics(y,p))\n        predictions=loaders[split].dataset.df[['file_name','video_id','label']].copy()\n        require(np.array_equal(predictions.label.to_numpy(),y),'Prediction order mismatch')\n        predictions['prediction']=p.argmax(1)\n        for c in range(len(classes)): predictions[f'prob_{c}']=p[:,c]\n        predictions.to_csv(folder/f'{split}_predictions.csv',index=False)\n    save_json(folder/'metrics.json',result)\n    save_json(folder/'complete.json',{'best_epoch':checkpoint['epoch'],'split_hash':protocol['split_hash']})\n    print(json.dumps(result,indent=2))\n    del model,optimizer,scheduler,scaler,checkpoint,loaders\n    gc.collect(); torch.cuda.empty_cache()\n\n\nif __name__ == '__main__':\n    settings = json.loads(Path(sys.argv[1]).read_text())\n    CONFIG = settings['config']\n    TUNE_OVERRIDES = settings['overrides']\n    OUTPUT_ROOT = Path(settings['output_root'])\n    SOURCES = settings['sources']\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    require(torch.cuda.device_count()==1, 'Worker must see exactly one assigned GPU')\n    task, name = settings['task'], sys.argv[2]\n    df = pd.read_csv(settings['manifest'])\n    print('Assigned CUDA device:',os.environ.get('CUDA_VISIBLE_DEVICES'),flush=True)\n    train_run(task,name,df)\n"

# Workers run as fresh Python processes: CUDA is masked before torch is imported.
# The embedded worker contains the model/data/training definitions from this notebook.
# Section 2 settings are passed explicitly so runtime configuration edits apply.
def launch_dual_gpu(task):
    import subprocess, sys, time, signal, fcntl
    require(torch.cuda.is_available() and torch.cuda.device_count() >= 2,
            'Select the Kaggle GPU T4 x2 accelerator. This launcher requires two GPUs.')
    directory = OUTPUT_ROOT / task
    directory.mkdir(parents=True, exist_ok=True)
    lock = (directory/'launcher.lock').open('w')
    try:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        lock.close()
        raise RuntimeError('A launcher is already running for this task/output folder.')
    active, failures = {}, []
    try:
        script = directory/'training_worker.py'
        script.write_text(WORKER_SOURCE)
        runtime = directory/'worker_settings.json'
        save_json(runtime, dict(config=CONFIG, overrides=TUNE_OVERRIDES,
            output_root=str(OUTPUT_ROOT.resolve()), sources=SOURCES, task=task,
            manifest=str((directory/'split_manifest.csv').resolve())))
        log_dir = directory/'logs'; log_dir.mkdir(exist_ok=True)
        require(len(MODEL_NAMES)==len(set(MODEL_NAMES)), 'Duplicate models in MODEL_NAMES')
        pending = list(MODEL_NAMES)
        require(all(n in REGISTRY for n in pending), 'Unknown model in MODEL_NAMES')
        visible = os.environ.get('CUDA_VISIBLE_DEVICES')
        gpu_tokens = visible.split(',') if visible else ['0','1']
        require(len(gpu_tokens)>=2, 'Two CUDA device identifiers are required')
        while pending or active:
            for gpu in (0,1):
                if gpu in active or not pending: continue
                name = pending.pop(0)
                log_path = log_dir/(name.replace('/','_')+'.log')
                log = log_path.open('a', buffering=1)
                env = os.environ.copy()
                env.update(CUDA_VISIBLE_DEVICES=gpu_tokens[gpu].strip(),
                    OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1',
                    PYTHONUNBUFFERED='1')
                try:
                    process = subprocess.Popen([sys.executable,str(script.resolve()),
                        str(runtime.resolve()),name], stdout=log,stderr=subprocess.STDOUT,
                        env=env,start_new_session=True)
                except BaseException:
                    log.close(); raise
                active[gpu] = (process,name,log,log_path)
                print(f'GPU {gpu}: {name} started. Log: {log_path}',flush=True)
            for gpu,(process,name,log,log_path) in list(active.items()):
                code = process.poll()
                if code is None: continue
                log.close(); del active[gpu]
                print(f'GPU {gpu}: {name} finished, exit code {code}',flush=True)
                if code:
                    failures.append(name)
                    print(log_path.read_text(errors='replace')[-4000:],flush=True)
                write_task_summary(task)
            if active: time.sleep(2)
    finally:
        # Stop both process groups (including loader children) if the cell is interrupted.
        for process,name,log,log_path in active.values():
            if process.poll() is None:
                try: os.killpg(process.pid,signal.SIGTERM)
                except ProcessLookupError: pass
        for process,name,log,log_path in active.values():
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try: os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError: pass
                process.wait()
            log.close()
        try: write_task_summary(task)
        finally: lock.close()
    require(not failures, 'Some models failed; inspect logs and rerun to resume: '+', '.join(failures))

if RUN_TRAINING:
    for task in TASKS: launch_dual_gpu(task)
else:
    for task in TASKS: write_task_summary(task)
    print('Dataset audit completed. Set RUN_TRAINING=True in section 2 to train on both GPUs.')


9. Results and optional download archive

Outputs: `/kaggle/working/outputs/transformers_severity_v2_two_lr/severity/`. Each model has its own subfolder. `best.pth` selects the epoch by validation loss; `best_model.json` selects the architecture by validation macro F1. The ZIP option includes partial runs and all resume checkpoints. This is a single patient-grouped split, not K-fold.


In [ ]:
from IPython.display import display, FileLink
import zipfile
for task in TASKS:
    summary = write_task_summary(task)
    if summary is not None: display(summary)
    print('Download the full output folder:', OUTPUT_ROOT / task)

# Optional archive: includes all saved outputs, including resume checkpoints.
# Disabled by default because an archive needs additional disk space.
# This cell can be run after a partial session; it does not launch training.
EXPORT_ZIP = False
if EXPORT_ZIP:
    archive = ROOT / (RUN_NAME + '.zip')
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as z:
        for path in sorted(OUTPUT_ROOT.rglob('*')):
            if path.is_file() and path.suffix != '.tmp':
                z.write(path, path.relative_to(OUTPUT_ROOT.parent))
    display(FileLink(str(archive.relative_to(ROOT))))
